# Quick Workflow Launcher

This notebook is a lightweight way to try the story-judge pipeline without manually editing workflow YAML files.

Change a few variables, generate a temporary workflow config, run it, and inspect the combined output. For larger batch experiments, use `python run_workflows.py` from the command line.

## 1. Setup

Run this cell first. The notebook must be opened from the `llm_as_story_judge` directory.

In [ ]:
from __future__ import annotations

from pathlib import Path
import yaml
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "run_workflows.py").exists():
    raise RuntimeError("Open this notebook from the llm_as_story_judge directory.")

CONFIG_DIR = ROOT / "configs"
NOTEBOOK_WORKFLOW = CONFIG_DIR / "notebook_workflow.yaml"
LLM_MANAGER_YAML = CONFIG_DIR / "llm_manager_config.yaml"
MODEL_KEYS_YAML = CONFIG_DIR / "model_key.yaml"

def load_yaml(path: str | Path):
    with Path(path).open("r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}

def save_yaml(path: str | Path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        yaml.safe_dump(data, sort_keys=False, allow_unicode=True),
        encoding="utf-8",
    )
    print(f"Saved {path.relative_to(ROOT)}")

print("Project root:", ROOT)
print("Available workflow configs:", [p.name for p in sorted(CONFIG_DIR.glob("*_workflow.yaml"))])

## 2. Choose A Quick Run Configuration

Edit only these values for a quick experiment. Use a small `BATCH_SIZE` when testing a model or prompt for the first time.

In [ ]:
MODEL_KEY = "kimi"
DATASET_PATH = "judge_data/dataset60.json"
EXP_NAME = "notebook_quick_test"
RUN_NAME = "demo"

PROMPTS_PATH = "baseline_assets/prompts/"
SCHEMAS_PATH = "baseline_assets/schemas/"

BATCH_SIZE = 5
MAX_RETRIES = 2
REASONING_ENABLED = True
VERBOSE_LLM = False
BASE_OUTPUT_DIR = "experiments"

## 3. Inspect Available Prompts And Schemas

This cell only prints the files used by the workflow. Edit the YAML files directly in your editor if you want to change prompts or output schemas.

In [ ]:
prompt_files = sorted((ROOT / PROMPTS_PATH).glob("*.yaml"))
schema_files = sorted((ROOT / SCHEMAS_PATH).glob("*.yaml"))

print("Prompts:")
for path in prompt_files:
    print(" -", path.relative_to(ROOT))

print("\nSchemas:")
for path in schema_files:
    print(" -", path.relative_to(ROOT))

## 4. Create The Notebook Workflow YAML

This cell writes `configs/notebook_workflow.yaml` from the variables above. The main launcher can consume it like any other workflow config.

In [ ]:
workflow_config = {
    "workflow": {
        "exp_name": EXP_NAME,
        "logical_model_key": MODEL_KEY,
        "dataset_path": DATASET_PATH,
        "schemas_path": SCHEMAS_PATH,
        "prompts_path": PROMPTS_PATH,
        "run_name": RUN_NAME,
        "batch_size": BATCH_SIZE,
        "max_retries": MAX_RETRIES,
        "reasoning_enabled": REASONING_ENABLED,
        "verbose_llm": VERBOSE_LLM,
        "base_output_dir": BASE_OUTPUT_DIR,
    }
}

save_yaml(NOTEBOOK_WORKFLOW, workflow_config)
workflow_config

## 5. Validate Before Running

This check catches the most common setup issues before any model call is made.

In [ ]:
model_keys = load_yaml(MODEL_KEYS_YAML)
checks = {
    "workflow_yaml": NOTEBOOK_WORKFLOW.exists(),
    "llm_manager_yaml": LLM_MANAGER_YAML.exists(),
    "model_keys_yaml": MODEL_KEYS_YAML.exists(),
    "model_key_exists": MODEL_KEY in model_keys,
    "dataset_exists": (ROOT / DATASET_PATH).exists(),
    "prompts_dir_exists": (ROOT / PROMPTS_PATH).exists(),
    "schemas_dir_exists": (ROOT / SCHEMAS_PATH).exists(),
    "has_prompt_files": len(prompt_files) > 0,
    "has_schema_files": len(schema_files) > 0,
}

for name, ok in checks.items():
    print(f"{name:20s} {'OK' if ok else 'MISSING'}")

if not all(checks.values()):
    missing = [name for name, ok in checks.items() if not ok]
    raise RuntimeError(f"Fix these checks before running: {missing}")

print("\nSelected model key:", MODEL_KEY, "->", model_keys[MODEL_KEY])

## 6. Run The Workflow

Set `RUN_WORKFLOW = True` only when you are ready to call the model.

In [ ]:
from agentic_forge.workflow_forge.structured_output_workflow_forge.structured_output_workflow_forge import StructuredOutputWorkflowForge
from agentic_forge.workflow_forge.structured_output_workflow_forge.combine_runs import combine_runs
from use_case_utils import DATASET_FIELD_BUILDERS, DefaultReasoningPolicy

RUN_WORKFLOW = False

if RUN_WORKFLOW:
    forge = StructuredOutputWorkflowForge.forge(
        llm_manager_yaml=LLM_MANAGER_YAML,
        model_keys_yaml=MODEL_KEYS_YAML,
        reasoning_policy=DefaultReasoningPolicy(),
    )

    workflow = forge.forge_structured_output_workflow(
        workflow_yaml_path=NOTEBOOK_WORKFLOW,
        field_builders=DATASET_FIELD_BUILDERS,
    )
    result = workflow.run()
    print("Run directory:", result["run_dir"])

    exp_dir = Path(BASE_OUTPUT_DIR) / EXP_NAME
    combine_runs(
        exp_dir,
        exp_dir / "combined_panel.csv",
        exp_dir / "combined_panel_meta.json",
    )
    print("Combined panel:", exp_dir / "combined_panel.csv")
else:
    print("Workflow not started. Set RUN_WORKFLOW = True to run it.")

## 7. Inspect Results

After a successful run, this cell loads the combined panel and shows the first rows.

In [ ]:
combined_panel = ROOT / BASE_OUTPUT_DIR / EXP_NAME / "combined_panel.csv"

if combined_panel.exists():
    df = pd.read_csv(combined_panel)
    print("Rows:", len(df))
    display(df.head())
else:
    print("No combined panel found yet:", combined_panel)